# Crash Data Analytics and Blackspot Identification

## G-TRISP Group A - Task 2

This project analyzes road accident data to identify:
- crash trends
- severity patterns
- temporal patterns
- spatial hotspots
- accident-prone blackspot regions

The workflow includes:
- data preprocessing
- exploratory data analysis (EDA)
- geospatial visualization
- DBSCAN clustering
- blackspot identification

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

import folium
from folium.plugins import HeatMap

from sklearn.cluster import DBSCAN

import warnings
warnings.filterwarnings("ignore")

from global_land_mask import globe

## Loading the Dataset

The crash dataset is loaded using Pandas for preprocessing and analysis.

In [2]:
df = pd.read_excel("../../datasets/accident_data.xlsx")

df.head()

,Accident_ID,District,Police_Station,Accident_DateTime,Latitude,Longitude,Road_Name,Road_Classification,Severity,No_of_Vehicles,...,Passengers_Minor_Injury,Pedestrians_Killed,Pedestrians_Grievous_Injury,Pedestrians_Minor_Injury,Collision_Type,Collision_Feature,Weather_Condition,Light_Condition,Visibility,Traffic_Violation
0,ACC00001,Ahmedabad,Naranpura PS,18-07-2022 07:14,20.571478,71.535809,Industrial Road,State Highway,Damage Only,1.0,...,0.0,0,0,0.0,Hit & Run,NaN,Clear,Night - Street Light On,Moderate,NaN
1,ACC00002,Junagadh,Keshod PS,23-08-2024 22:34,22.166362,73.576592,NH-8,Urban Road,Minor,3.0,...,0.0,0,0,1.0,Pedestrian Hit,Bridge,Foggy,Night - No Street Light,Moderate,Wrong Side Driving
2,ACC00003,Ahmedabad,Chandkheda PS,06-12-2022 19:16,23.456766,74.230629,Ring Road,National Highway,Grievous,1.0,...,1.0,0,0,0.0,Head-On,Flyover,NaN,Night - No Street Light,Moderate,Drunk Driving
3,ACC00004,Anand,Khambhat PS,18-07-2023 20:53,21.802832,69.781960,City Road,National Highway,Damage Only,4.0,...,0.0,0,0,0.0,Single Vehicle,Pothole Area,Rainy,Night - Street Light On,Moderate,Overloading
4,ACC00005,Gandhinagar,Dehgam PS,26-03-2022 21:20,23.669600,68.302232,SH-20,District Road,Minor,4.0,...,2.0,0,0,NaN,Hit & Run,Curve,Hazy,Night - No Street Light,Moderate,NaN


## Dataset Overview

The dataset contains accident-level records including:
- accident location
- severity
- road classification
- weather conditions
- collision details
- injury statistics

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Accident_ID                  1500 non-null   object 
 1   District                     1500 non-null   object 
 2   Police_Station               1500 non-null   object 
 3   Accident_DateTime            1500 non-null   object 
 4   Latitude                     1463 non-null   float64
 5   Longitude                    1473 non-null   float64
 6   Road_Name                    1500 non-null   object 
 7   Road_Classification          1500 non-null   object 
 8   Severity                     1500 non-null   object 
 9   No_of_Vehicles               1410 non-null   float64
 10  Drivers_Killed               1500 non-null   int64  
 11  Drivers_Grievous_Injury      1410 non-null   float64
 12  Drivers_Minor_Injury         1500 non-null   int64  
 13  Passengers_Killed 

## Statistical Summary

A statistical overview helps understand the numerical columns and their distributions.

In [4]:
df.describe()

,Latitude,Longitude,No_of_Vehicles,Drivers_Killed,Drivers_Grievous_Injury,Drivers_Minor_Injury,Passengers_Killed,Passengers_Grievous_Injury,Passengers_Minor_Injury,Pedestrians_Killed,Pedestrians_Grievous_Injury,Pedestrians_Minor_Injury
count,1463.000000,1473.000000,1410.000000,1500.00000,1410.000000,1500.000000,1500.000000,1500.000000,1410.000000,1500.000000,1500.000000,1410.000000
mean,22.428336,71.206824,2.509929,0.14800,0.638298,1.130667,0.142667,0.368667,0.774468,0.080000,0.194667,0.390780
std,1.332930,1.802032,1.129123,0.47425,0.998769,1.044184,0.471658,0.688768,0.819511,0.271384,0.396076,0.488098
min,20.105477,68.104306,1.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,21.315576,69.681136,1.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,22.477519,71.250296,3.000000,0.00000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
75%,23.580748,72.714466,4.000000,0.00000,1.000000,2.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
max,24.698031,74.385431,4.000000,2.00000,3.000000,3.000000,2.000000,2.000000,2.000000,1.000000,1.000000,1.000000


## Missing Value Analysis

The dataset contains missing values in several columns such as:
- coordinates
- weather conditions
- traffic violations

Different preprocessing strategies will be applied depending on the type and importance of the data.

In [5]:
df.isnull().sum()

Accident_ID                      0
District                         0
Police_Station                   0
Accident_DateTime                0
Latitude                        37
Longitude                       27
Road_Name                        0
Road_Classification              0
Severity                         0
No_of_Vehicles                  90
Drivers_Killed                   0
Drivers_Grievous_Injury         90
Drivers_Minor_Injury             0
Passengers_Killed                0
Passengers_Grievous_Injury       0
Passengers_Minor_Injury         90
Pedestrians_Killed               0
Pedestrians_Grievous_Injury      0
Pedestrians_Minor_Injury        90
Collision_Type                   0
Collision_Feature               88
Weather_Condition              249
Light_Condition                  0
Visibility                     393
Traffic_Violation              402
dtype: int64

## Datetime Conversion

The `Accident_DateTime` column is converted into datetime format to enable:
- hourly analysis
- monthly trends
- temporal feature extraction

In [6]:
df['Accident_DateTime'] = pd.to_datetime(
    df['Accident_DateTime'],
    dayfirst=True,
    errors='coerce'
)

df['Hour'] = df['Accident_DateTime'].dt.hour
df['Month'] = df['Accident_DateTime'].dt.month
df['Year'] = df['Accident_DateTime'].dt.year

## Handling Missing Coordinates

Rows with missing latitude or longitude values are removed because spatial analysis and clustering require valid geographic coordinates.

In [7]:
df = df.dropna(subset=['Latitude', 'Longitude'])

## Geographic Coordinate Validation

Some accident coordinates appeared over ocean regions, indicating possible synthetic or randomized spatial data.

To improve spatial realism, land/ocean validation was applied using the `global-land-mask` library. Coordinates identified over water bodies were removed prior to spatial clustering and hotspot analysis.

In [8]:
df['Is_Land'] = df.apply(
    lambda row: globe.is_land(
        row['Latitude'],
        row['Longitude']
    ),
    axis=1
)

df = df[df['Is_Land']]

## Handling Missing Numerical Values

Missing numerical injury-related values are replaced with `0` to maintain consistency in quantitative analysis.

In [9]:
numerical_cols = [
    'No_of_Vehicles',
    'Drivers_Grievous_Injury',
    'Passengers_Minor_Injury',
    'Pedestrians_Minor_Injury'
]

for col in numerical_cols:
    df[col] = df[col].fillna(0)

## Handling Missing Categorical Values

Missing categorical values are replaced with `"Unknown"` to preserve accident records while maintaining interpretability.

In [10]:
categorical_cols = [
    'Collision_Feature',
    'Weather_Condition',
    'Visibility',
    'Traffic_Violation'
]

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

## Cleaned Dataset Preview

The dataset after preprocessing and feature engineering.

In [11]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1043 entries, 1 to 1496
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Accident_ID                  1043 non-null   object        
 1   District                     1043 non-null   object        
 2   Police_Station               1043 non-null   object        
 3   Accident_DateTime            1043 non-null   datetime64[ns]
 4   Latitude                     1043 non-null   float64       
 5   Longitude                    1043 non-null   float64       
 6   Road_Name                    1043 non-null   object        
 7   Road_Classification          1043 non-null   object        
 8   Severity                     1043 non-null   object        
 9   No_of_Vehicles               1043 non-null   float64       
 10  Drivers_Killed               1043 non-null   int64         
 11  Drivers_Grievous_Injury      1043 non-null   flo

## Severity Distribution Analysis

This analysis shows the distribution of accident severity categories.

In [12]:
severity_counts = df['Severity'].value_counts()

fig = px.bar(
    x=severity_counts.index,
    y=severity_counts.values,
    labels={'x': 'Severity', 'y': 'Count'},
    title='Accident Severity Distribution'
)

fig.show()

## District-wise Accident Analysis

This analysis identifies districts with the highest accident frequency.

In [13]:
district_counts = df['District'].value_counts().head(10)

fig = px.bar(
    x=district_counts.index,
    y=district_counts.values,
    labels={'x': 'District', 'y': 'Accident Count'},
    title='Top 10 Districts by Accident Count'
)

fig.show()

## Hourly Accident Trend

This analysis shows how accident frequency varies throughout the day.

In [14]:
hourly_counts = df['Hour'].value_counts().sort_index()

fig = px.line(
    x=hourly_counts.index,
    y=hourly_counts.values,
    labels={'x': 'Hour of Day', 'y': 'Accident Count'},
    title='Hourly Accident Distribution'
)

fig.show()

## Monthly Accident Trend

Monthly analysis helps identify seasonal accident patterns.

In [15]:
monthly_counts = df['Month'].value_counts().sort_index()

fig = px.line(
    x=monthly_counts.index,
    y=monthly_counts.values,
    labels={'x': 'Month', 'y': 'Accident Count'},
    title='Monthly Accident Distribution'
)

fig.show()

## Weather Condition Analysis

This analysis examines accident occurrence under different weather conditions.

In [16]:
weather_counts = df['Weather_Condition'].value_counts()

fig = px.bar(
    x=weather_counts.index,
    y=weather_counts.values,
    labels={'x': 'Weather Condition', 'y': 'Count'},
    title='Accidents by Weather Condition'
)

fig.show()

## Traffic Violation Analysis

This analysis identifies the most common traffic violations associated with accidents.

In [17]:
violation_counts = df['Traffic_Violation'].value_counts().head(10)

fig = px.bar(
    x=violation_counts.index,
    y=violation_counts.values,
    labels={'x': 'Violation', 'y': 'Count'},
    title='Traffic Violation Distribution'
)

fig.show()

## Spatial Visualization of Accidents

Accident locations are visualized using geographic coordinates to identify spatial concentration patterns.

In [18]:
m = folium.Map(
    location=[23.0, 72.5],
    zoom_start=7
)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=2,
        fill=True
    ).add_to(m)

m

## Accident Density Heatmap

Heatmaps help visualize regions with high accident concentration.

In [19]:
heat_data = df[['Latitude', 'Longitude']].values.tolist()

m = folium.Map(
    location=[23.0, 72.5],
    zoom_start=7
)

HeatMap(heat_data).add_to(m)

m

## DBSCAN Clustering

DBSCAN groups nearby accident locations into clusters based on spatial density.

Clusters may represent accident-prone hotspot regions.

In [20]:
coords = df[['Latitude', 'Longitude']]

dbscan = DBSCAN(
    eps=0.1,
    min_samples=3
)

df['Cluster'] = dbscan.fit_predict(coords)

df[['Latitude', 'Longitude', 'Cluster']].head()

,Latitude,Longitude,Cluster
1,22.166362,73.576592,-1
2,23.456766,74.230629,-1
3,21.802832,69.781960,12
4,23.669600,68.302232,0
5,21.308611,73.756284,-1


## Cluster Distribution

This analysis shows the number of accidents identified within each cluster.

In [21]:
cluster_counts = df['Cluster'].value_counts()

fig = px.bar(
    x=cluster_counts.index.astype(str),
    y=cluster_counts.values,
    labels={'x': 'Cluster ID', 'y': 'Accident Count'},
    title='DBSCAN Cluster Distribution'
)

fig.show()

## Blackspot Identification

Blackspots are identified by ranking clusters using:
- accident frequency
- fatalities
- grievous injuries

In [22]:
cluster_summary = df.groupby('Cluster').agg({
    'Accident_ID': 'count',
    'Drivers_Killed': 'sum',
    'Drivers_Grievous_Injury': 'sum'
}).reset_index()

cluster_summary.columns = [
    'Cluster',
    'Accident_Count',
    'Drivers_Killed',
    'Drivers_Grievous_Injury'
]

cluster_summary['Severity_Score'] = (
    cluster_summary['Drivers_Killed'] * 5 +
    cluster_summary['Drivers_Grievous_Injury'] * 3
)

blackspots = cluster_summary.sort_values(
    by='Severity_Score',
    ascending=False
)

blackspots.head(10)

,Cluster,Accident_Count,Drivers_Killed,Drivers_Grievous_Injury,Severity_Score
0,-1,421,48,222.0,906.0
3,2,12,5,14.0,67.0
5,4,8,6,9.0,57.0
89,88,12,0,16.0,48.0
87,86,7,0,15.0,45.0
29,28,10,3,8.0,39.0
34,33,11,0,13.0,39.0
20,19,11,2,9.0,37.0
18,17,8,1,10.0,35.0
123,122,3,5,3.0,34.0


## Spatial Data Limitation

During spatial analysis, several accident coordinates appeared uniformly distributed across the geographic region, including locations near coastal and non-road regions.

As a result:
- DBSCAN clustering produced limited meaningful hotspot formation
- many points were classified as noise
- heatmap density appeared broadly uniform

The implemented spatial analysis pipeline remains fully functional and reproducible for real-world crash datasets containing realistic coordinate distributions.